In [ ]:
# Lab type: extend
# Course: AI401 — AI Applications with LLMs
# Lesson: Testing LLM-Powered Systems: The Evaluations Framework
# Task: Extend a working evaluation framework with prompt version tracking and a regression gate

# Lab: Extending the Evaluations Framework

The baseline below gives you a working golden-set evaluator and a `PromptRegistry` that stores prompts by name — but has no version tracking and no regression gate.

**Your task:** Implement three extensions:

1. Add SHA-256 version tracking to `PromptRegistry`
2. Implement `regression_gate()` — raises `RegressionError` if accuracy drops too far
3. Implement `compare_prompt_versions()` — returns per-example accuracy deltas between two prompt hashes

## Baseline (working — do not modify)

In [ ]:
import hashlib
import json
from pathlib import Path
import tempfile


# ---------------------------------------------------------------------------
# Minimal golden set (no live API needed — mock pipeline used in tests)
# ---------------------------------------------------------------------------

GOLDEN_SET = [
    {"id": "t001", "input": "Payment failed three times — urgent",
     "expected": {"category": "billing", "priority": 4}},
    {"id": "t002", "input": "Tracking number shows delivered but package not received",
     "expected": {"category": "shipping", "priority": 3}},
    {"id": "t003", "input": "App crashes on login since last update",
     "expected": {"category": "technical", "priority": 4}},
    {"id": "t004", "input": "Invoice date shows 2031 — clearly wrong",
     "expected": {"category": "billing", "priority": 2}},
    {"id": "t005", "input": "How do I change my email address?",
     "expected": {"category": "other", "priority": 1}},
]


# ---------------------------------------------------------------------------
# Mock pipeline — simulates model output without a live API call
# ---------------------------------------------------------------------------

class MockOutput:
    def __init__(self, category: str, priority: int):
        self.category = category
        self.priority = priority


def make_mock_pipeline(accuracy: float):
    """Return a mock pipeline that gets `accuracy` fraction of golden set right."""
    correct = {item["id"]: item["expected"] for item in GOLDEN_SET}
    ids_in_order = [item["id"] for item in GOLDEN_SET]
    n_correct = int(len(ids_in_order) * accuracy)

    call_count = [0]

    def pipeline(ticket_text: str) -> MockOutput:
        idx = call_count[0] % len(ids_in_order)
        tid = ids_in_order[idx]
        call_count[0] += 1
        if idx < n_correct:
            exp = correct[tid]
            return MockOutput(exp["category"], exp["priority"])
        return MockOutput("other", 1)  # Wrong answer
    return pipeline


# ---------------------------------------------------------------------------
# Golden set evaluator
# ---------------------------------------------------------------------------

def evaluate_golden_set(pipeline_fn, golden_set: list) -> dict:
    """
    Run pipeline_fn over golden_set, return accuracy report.
    """
    results = {"total": 0, "passed": 0, "failed": [], "per_id": {}}
    for item in golden_set:
        results["total"] += 1
        output = pipeline_fn(item["input"])
        expected = item["expected"]
        passed = (
            output.category == expected["category"]
            and output.priority == expected["priority"]
        )
        results["per_id"][item["id"]] = passed
        if passed:
            results["passed"] += 1
        else:
            results["failed"].append({"id": item["id"], "expected": expected,
                                       "got": {"category": output.category,
                                               "priority": output.priority}})
    results["accuracy"] = results["passed"] / results["total"]
    return results


# ---------------------------------------------------------------------------
# PromptRegistry — stores prompts by name (NO VERSION TRACKING YET)
# ---------------------------------------------------------------------------

class PromptRegistry:
    """
    Stores prompts by name.
    TODO: Add SHA-256 version tracking in Extension 1.
    """
    def __init__(self, registry_path: Path):
        self.registry_path = registry_path
        if registry_path.exists():
            self._data: dict = json.loads(registry_path.read_text())
        else:
            self._data = {}

    def register(self, name: str, prompt: str, accuracy: float) -> None:
        """Register a prompt and its golden-set accuracy."""
        self._data[name] = {"prompt": prompt, "accuracy": accuracy}
        self.registry_path.write_text(json.dumps(self._data, indent=2))

    def get_accuracy(self, name: str) -> float | None:
        """Return the registered accuracy for a prompt name, or None."""
        return self._data.get(name, {}).get("accuracy")


# Quick smoke-test of baseline
pipeline_perfect = make_mock_pipeline(1.0)
report = evaluate_golden_set(pipeline_perfect, GOLDEN_SET)
print(f"Baseline smoke-test — accuracy: {report['accuracy']:.0%}")

## Extension 1: Add version tracking to `PromptRegistry`

Extend `register()` to also store a SHA-256 hash of the prompt text (first 12 characters of the hex digest). Add a `get_registered_hash(name)` method that returns the stored hash or `None`.

In [ ]:
class PromptRegistryV2(PromptRegistry):
    """
    PromptRegistry with SHA-256 version tracking.

    Extend register() to store a prompt_hash field.
    Add get_registered_hash(name) -> str | None.
    """

    def register(self, name: str, prompt: str, accuracy: float) -> str:
        """Register prompt + accuracy. Return the prompt hash."""
        pass  # implement here

    def get_registered_hash(self, name: str) -> str | None:
        """Return the registered hash for name, or None if not found."""
        pass  # implement here


# Verify your implementation
with tempfile.TemporaryDirectory() as tmpdir:
    reg = PromptRegistryV2(Path(tmpdir) / 'registry.json')
    prompt_v1 = 'Classify as: billing, technical, shipping, or other. Return one word.'
    h = reg.register('ticket_classifier', prompt_v1, 0.96)
    print(f'Registered hash: {h!r} (should be 12-char hex string)')
    assert h is not None and len(h) == 12, 'Hash must be a 12-character hex string'
    assert reg.get_registered_hash('ticket_classifier') == h
    assert reg.get_registered_hash('nonexistent') is None
    print('Extension 1: PASS')

## Extension 2: Implement `regression_gate()`

The gate should:
- Run `evaluate_golden_set()` against the current prompt
- Compare current accuracy to the registered baseline accuracy
- Raise `RegressionError` if the accuracy drop exceeds `threshold` (default 0.02)
- Update the registry with the new accuracy if the gate passes
- Skip the check (and log a message) if the prompt hash is unchanged

In [ ]:
class RegressionError(AssertionError):
    """Raised when a prompt change causes accuracy to drop beyond the threshold."""


def regression_gate(
    pipeline_fn,
    golden_set: list,
    registry: PromptRegistryV2,
    prompt_name: str,
    current_prompt: str,
    threshold: float = 0.02,
) -> dict:
    """
    Check that current_prompt has not caused an accuracy regression.
    Returns the accuracy report dict.
    Raises RegressionError if accuracy dropped by more than threshold.
    Skips evaluation (returns None) if hash is unchanged.
    """
    pass  # implement here


# Verify your implementation
with tempfile.TemporaryDirectory() as tmpdir:
    reg = PromptRegistryV2(Path(tmpdir) / 'registry.json')
    prompt = 'Classify as: billing, technical, shipping, or other.'
    reg.register('tc', prompt, 0.96)

    # Gate should pass: new pipeline has 100% accuracy (no regression)
    result = regression_gate(make_mock_pipeline(1.0), GOLDEN_SET, reg, 'tc', prompt + ' Return one word.')
    print(f'Gate passed — accuracy: {result["accuracy"]:.0%}')

    # Gate should fail: new pipeline has 40% accuracy (drop > 0.02)
    try:
        regression_gate(make_mock_pipeline(0.4), GOLDEN_SET, reg, 'tc', prompt + ' Be concise.')
        print('FAIL: RegressionError should have been raised')
    except RegressionError as e:
        print(f'RegressionError raised correctly: {e}')
    print('Extension 2: PASS')

## Extension 3: Implement `compare_prompt_versions()`

Given two prompt hashes and corresponding pipelines, return a dict showing which golden-set examples changed between the two versions:

```python
{
  'v1_accuracy': 0.80,
  'v2_accuracy': 1.00,
  'delta': 0.20,
  'regressions': ['t003', 't004'],   # passed in v1, failed in v2
  'improvements': ['t001'],           # failed in v1, passed in v2
}
```

In [ ]:
def compare_prompt_versions(
    pipeline_v1,
    pipeline_v2,
    golden_set: list,
) -> dict:
    """
    Compare two pipelines on the golden set.
    Returns a dict with v1_accuracy, v2_accuracy, delta,
    regressions (ids that went from pass to fail), and
    improvements (ids that went from fail to pass).
    """
    pass  # implement here


# Verify your implementation
report = compare_prompt_versions(
    make_mock_pipeline(0.8),   # 80% accurate (4/5 correct)
    make_mock_pipeline(1.0),   # 100% accurate
    GOLDEN_SET,
)
print('v1_accuracy :', report.get('v1_accuracy'))
print('v2_accuracy :', report.get('v2_accuracy'))
print('delta       :', report.get('delta'))
print('regressions :', report.get('regressions'))
print('improvements:', report.get('improvements'))
assert report.get('delta', 0) > 0, 'v2 should be more accurate than v1'
print('Extension 3: PASS')